In [ ]:
import logging
import re
from collections import Counter
import pandas as pd
from utils import standardize_column_names

# Configure logging to show in notebook cells
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', force=True)


In [ ]:
# utils.py
# --- Function: Standardize Column Names ---
def standardize_column_names(df):
    """
    Standardizes DataFrame column names: lowercase, underscores, and alphanumeric only.
    """
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[ \-]+", "_", regex=True)
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

# --- Function: Filter to County-Level Rows ---
def filter_to_county_level(df):
    """
    Removes state-level and summary rows based on FIPS and county name content.
    """
    for fips_col in ['fips', 'fips_code', 'fipstxt']:
        if fips_col in df.columns:
            df = df[df[fips_col].astype(str).str[-3:] != '000']
    if 'county' in df.columns:
        df = df[~df['county'].str.contains("state|total|arkansas", case=False, na=False)]
    return df

# --- Function: Log Duplicate County-Attribute Pairs ---
def log_duplicate_attributes(df, key="unknown"):
    """
    Logs a warning if duplicate (county, attribute) pairs are found.
    """
    if df.duplicated(subset=['county', 'attribute']).any():
        logging.warning(f"⚠️ {key} has duplicate county-attribute pairs.")

# --- Function: Extract Year Frequencies from Column Names ---
def extract_common_year_from_columns(df, return_all=False, min_freq=1):
    """
    Extracts most common 4-digit years from column names.
    
    Args:
        df (pd.DataFrame): DataFrame to analyze.
        return_all (bool): If True, return all year frequencies.
        min_freq (int): Minimum count for inclusion if returning all.

    Returns:
        str or dict: Most common year (str), or dict of {year: count}
    """
    year_pattern = re.compile(r'(19|20)\d{2}')
    years = []

    for col in df.columns:
        matches = year_pattern.findall(col)
        full_matches = re.findall(r'(19|20)\d{2}', col)
        years.extend(full_matches)

    year_counts = Counter(years)

    if return_all:
        return {year: count for year, count in year_counts.items() if count >= min_freq}
    elif year_counts:
        return year_counts.most_common(1)[0][0]
    else:
        return None

# --- Function: Subset DataFrame Columns to a Given Year ---
def subset_columns_by_year(df, year):
    """
    Returns a DataFrame with only the columns that contain the specified year.
    """
    return df[[col for col in df.columns if str(year) in col]].copy()

# --- Constant: North Central Arkansas Counties ---
nca_counties = [
    'baxter', 'cleburne', 'fulton', 'independence', 'izard', 'jackson',
    'marion', 'searcy', 'sharp', 'stone', 'van buren', 'white', 'woodruff'
]

# --- Logging confirmation ---
logging.info("✅ Utility functions loaded from utils.py")


INFO: ✅ Utility functions loaded from utils.py


In [5]:
if __name__ == "__main__":
    # Configure logging if running this file directly
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    # Sample test DataFrame for column functions
    import pandas as pd

    test_df = pd.DataFrame({
        'FIPS Code': [1001, 1003, 1005],
        'Area Name': ['Autauga County, AL', 'Baldwin County, AL', 'Barbour County, AL'],
        'Attribute': ['population_2023', 'population_2023', 'population_2023'],
        'Value': [55869, 233390, 24686]
    })

    logging.info("🔍 Testing standardize_column_names...")


INFO: 🔍 Testing standardize_column_names...


In [6]:
df = pd.DataFrame({
    "Area Name": ["Baxter County, AR", "Fulton County, AR"],
    "Attribute": ["Population_2023", "Population_2023"]
})

df = standardize_column_names(df)
logging.info("✅ Column names standardized.")
print(df.columns.tolist())

INFO: ✅ Column names standardized.


['area_name', 'attribute']
